In [1]:
import re
import os
import sys
import json
sys.path.append('../scripts')

from bs4 import BeautifulSoup
from utils import ITEM_PATTERNS

In [2]:
with open('../cfg.json', 'r') as f:
    config = json.load(f)
read_folder_path = config['raw_data_folder']
write_folder_path = config['processed_data_folder']
# create the destination folder if it doesn't exist
os.makedirs(write_folder_path, exist_ok=True)

In [3]:
# turn tables into meaningful text
def table_to_text(table) -> str:
    rows = table.find_all("tr")
    if not rows or len(rows) < 1:
        return ""

    grid = []
    for row in rows:
        cells = row.find_all(["th", "td"])
        grid.append([clean(c.get_text()) for c in cells])

    # remove empty rows
    grid = [r for r in grid if any(c for c in r)]
    if len(grid) < 1:
        return ""

    # make sure we group $ and % with numbers
    for row in grid:
        i = 0
        while i < len(row) - 1:
            if row[i] == "$" and row[i + 1]:
                row[i] = "$" + row[i + 1]
                row.pop(i + 1)
            elif row[i + 1] == "%" and row[i]:
                row[i] = row[i] + "%"
                row.pop(i + 1)
            else:
                i += 1

    # normalize number of cols
    max_cols = max(len(r) for r in grid)
    grid = [r + [""] * (max_cols - len(r)) for r in grid]

    lines = []
    for row in grid:
        # make sure we're not getting rid of any ITEM title
        row_text = " ".join(row)
        is_item_header = re.search(r"ITEM\s+\d+[A-Z]?\.?", row_text, re.IGNORECASE)
        
        if is_item_header:
            lines.append(row_text.strip())
            continue

        values = [c for c in row if c and c not in {"—", "–", " "}]
        if not values:
            continue

        # if the first column might be a label, mark it as such
        label = row[0] if row[0] and not re.match(r"^[\d\,\.\-\(\)%]*$", row[0]) else ""
        data_values = values[1:] if label else values

        if label:
            lines.append(f"{label}: " + " | ".join(data_values))
        elif data_values:
            lines.append(" | ".join(data_values))

    return "\n".join(lines) if lines else ""

# clean individual cells in tables
def clean(cell: str) -> str:
    cell = cell.replace("\xa0", " ")
    cell = re.sub(r"\s+", " ", cell)
    cell = cell.strip()
    return cell

# preprocess raw 10-K doc
def process_doc(doc) -> str:
    soup = BeautifulSoup(doc, 'lxml')

    for tag in soup(["script", "style"]):
        tag.decompose()

    for table in soup.find_all("table"):
        table_text = table_to_text(table)
        replacement = soup.new_string("\n" + table_text + "\n")
        table.replace_with(replacement)

    # return soup.prettify(formatter=lambda s: s.replace(u'\xa0', ' '))
    return soup.get_text(separator="\n").replace(u'\xa0', ' ')

In [4]:
# go through filings
for root, dirs, files in os.walk(read_folder_path):
    for file_name in files:
        if file_name.endswith('.txt'):
            file_path = os.path.join(root, file_name)
            # get ticker
            parts = file_path.split(os.sep)
            if len(parts) >= 3:
                print(parts)
                ticker = parts[-4].split('/')[-1] 
                part = parts[-2] 
                with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                    content = f.read()
                    print(f"\n{'='*70}")
                    print(f"Processing {ticker} | {part}")
                    print(f"{'='*70}")
                    
                    # only extract 10-K part
                    docs = re.findall(
                        r"<DOCUMENT>.*?<TYPE>\s*10-K.*?<TEXT>(.*?)</TEXT>.*?</DOCUMENT>",
                        content,
                        flags=re.IGNORECASE | re.DOTALL
                    )
                    
                    if not docs:
                        print("No 10-K document found")
                        continue
                    
                    doc = docs[0]
                    processed_text = process_doc(doc)
                    
                    # check if we find all the needed item patterns (or the majority)
                    found_items = {}
                    for item_key, pattern in ITEM_PATTERNS.items():
                        matches = list(re.finditer(pattern, processed_text, re.IGNORECASE))
                        if matches:
                            found_items[item_key] = len(matches)
                    
                    print(f"✓ Found {len(found_items)} items:")
                    for item_key in sorted(found_items.keys()):
                        count = found_items[item_key]
                        print(f"  {item_key}: {count} match(es)")
                    with open(write_folder_path + f"{ticker}_{part}.txt", 'w', encoding='utf-8') as out_f:
                        out_f.write(processed_text)


['../data/sec-edgar-filings/AAPL', '10-K', '0000320193-24-000123', 'full-submission.txt']

Processing AAPL | 0000320193-24-000123
✓ Found 20 items:
  item_1: 2 match(es)
  item_10: 2 match(es)
  item_11: 2 match(es)
  item_12: 2 match(es)
  item_13: 2 match(es)
  item_14: 2 match(es)
  item_15: 2 match(es)
  item_1a: 2 match(es)
  item_1b: 2 match(es)
  item_2: 2 match(es)
  item_3: 2 match(es)
  item_4: 2 match(es)
  item_5: 2 match(es)
  item_6: 2 match(es)
  item_7: 2 match(es)
  item_7a: 2 match(es)
  item_8: 2 match(es)
  item_9: 2 match(es)
  item_9a: 2 match(es)
  item_9b: 2 match(es)
['../data/sec-edgar-filings/AAPL', '10-K', '0000320193-25-000079', 'full-submission.txt']

Processing AAPL | 0000320193-25-000079
✓ Found 20 items:
  item_1: 2 match(es)
  item_10: 2 match(es)
  item_11: 2 match(es)
  item_12: 2 match(es)
  item_13: 2 match(es)
  item_14: 2 match(es)
  item_15: 2 match(es)
  item_1a: 2 match(es)
  item_1b: 2 match(es)
  item_2: 2 match(es)
  item_3: 2 match(es)
  i